# CLM-0.4 Release — 1M smoke → 30M
Run the complete 1M engineering smoke first. The 30M section is deliberately disabled until the smoke emits `READY_FOR_30M`.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, torch
WORK = Path('/kaggle/working')
ROOT = WORK / 'mini-cells'
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','switch','main'], cwd=ROOT, check=True)
    subprocess.run(['git','pull','--ff-only','origin','main'], cwd=ROOT, check=True)
os.chdir(ROOT)
HEAD = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
print('HEAD', HEAD)
print('GPUs', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, 'CLM-0.4 Release expects T4 x2'

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-e',str(ROOT)+'[lm]'], check=True)
sys.path.insert(0, str(ROOT / 'src')) if str(ROOT / 'src') not in sys.path else None
subprocess.run([sys.executable, str(ROOT/'scripts/research/run.py'), 'clm-0.4-release', '--profile', 'smoke-1m', '--plan-only'], cwd=ROOT, check=True)

## Stage A — full 1M engineering smoke

In [ ]:
REVISION = 'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64'
SMOKE_DATA = WORK / 'clm-0.4-release-1m-data'
if not (SMOKE_DATA/'release-profile.json').is_file():
    subprocess.run([sys.executable, str(ROOT/'scripts/research/prepare_clm_0_4_release_data.py'), '--profile', 'smoke-1m', '--dataset-revision', REVISION, '--out', str(SMOKE_DATA)], cwd=ROOT, check=True)
print(json.dumps(json.loads((SMOKE_DATA/'release-profile.json').read_text()), indent=2, sort_keys=True))

In [ ]:
SMOKE_OUT = WORK / 'clm-0.4-release-1m'
subprocess.run([sys.executable, str(ROOT/'scripts/research/run.py'), 'clm-0.4-release', '--profile', 'smoke-1m', '--data-dir', str(SMOKE_DATA), '--out', str(SMOKE_OUT), '--device', 'cuda', '--devices', 'cuda:0,cuda:1'], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT/'scripts/research/report.py'), 'clm-0.4-release', '--results', str(SMOKE_OUT)], cwd=ROOT, check=True)
READINESS = SMOKE_OUT / 'release-readiness.json'
readiness = json.loads(READINESS.read_text())
print(json.dumps(readiness, indent=2, sort_keys=True))
assert readiness['status'] == 'READY_FOR_30M'

In [ ]:
PUSH_SMOKE_RESULTS = True
if PUSH_SMOKE_RESULTS:
    subprocess.run([sys.executable, str(ROOT/'scripts/research/publish.py'), 'clm-0.4-release', '--results', str(SMOKE_OUT), '--push'], cwd=ROOT, check=True)
print((SMOKE_OUT/'RESULTS.md').read_text())

## Stage B — 30M release
Inspect the complete 1M results first. Only then set `START_30M = True`. If any critical release source changed since Stage A, the runner will reject the 30M launch.

In [ ]:
START_30M = False
if not START_30M:
    print('30M is intentionally blocked. Inspect the 1M smoke, then set START_30M=True.')

In [ ]:
if START_30M:
    RELEASE_DATA = WORK / 'clm-0.4-release-30m-data'
    if not (RELEASE_DATA/'release-profile.json').is_file():
        subprocess.run([sys.executable, str(ROOT/'scripts/research/prepare_clm_0_4_release_data.py'), '--profile', 'release-30m', '--dataset-revision', REVISION, '--out', str(RELEASE_DATA)], cwd=ROOT, check=True)
    print(json.dumps(json.loads((RELEASE_DATA/'release-profile.json').read_text()), indent=2, sort_keys=True))

In [ ]:
if START_30M:
    RELEASE_OUT = WORK / 'clm-0.4-release-30m'
    subprocess.run([sys.executable, str(ROOT/'scripts/research/run.py'), 'clm-0.4-release', '--profile', 'release-30m', '--data-dir', str(RELEASE_DATA), '--out', str(RELEASE_OUT), '--smoke-readiness', str(READINESS), '--device', 'cuda', '--devices', 'cuda:0,cuda:1'], cwd=ROOT, check=True)
    subprocess.run([sys.executable, str(ROOT/'scripts/research/report.py'), 'clm-0.4-release', '--results', str(RELEASE_OUT)], cwd=ROOT, check=True)
    print((RELEASE_OUT/'RESULTS.md').read_text())

In [ ]:
PUSH_30M_RESULTS = True
if START_30M and PUSH_30M_RESULTS:
    subprocess.run([sys.executable, str(ROOT/'scripts/research/publish.py'), 'clm-0.4-release', '--results', str(RELEASE_OUT), '--push'], cwd=ROOT, check=True)